# Generate Week 3.5 and Week 4 Reports from Results

This notebook reads the saved result files for Week 3.5 and Week 4, builds summary tables, and writes Markdown reports into the `reports` folder.

By default, it writes generated files with `generated-from-results` in the filename so existing manually written reports are not overwritten. Change `WRITE_CANONICAL_REPORTS = True` in the configuration cell if you want the notebook to replace `week-03-5-report.md` and `week-04-report.md`.

## What This Notebook Uses

Week 3.5 inputs:

- `imports/2026-06-14-week3.5/qwen05_high_accuracy_baseline/metrics.json`
- `Week 3.5/results/reference_successful_run/results/percentage_summary.csv`

Week 4 inputs:

- `imports/2026-06-14-week4/Week 4/results/gradient_ascent_unlearning_v1/results/metrics.json`
- `imports/2026-06-14-week4/Week 4/results/gradient_ascent_unlearning_v1/results/percentage_summary.csv`
- `imports/2026-06-14-week4/Week 4/results/gradient_ascent_unlearning_v1/results/unlearning_history.csv`
- `analysis/week3.5_vs_week4_results_2026-06-14.md`, if available

The notebook is intentionally focused on Week 3.5 and Week 4 because those are the phases with direct learning and unlearning result artifacts.

In [ ]:
import csv
import json
import math
# GitHub-backed Colab setup. Keep GITHUB_TOKEN in Colab Secrets.
from pathlib import Path
import sys
import urllib.request

HELPER_PATH = Path('/content/github_colab_sync.py')
HELPER_URL = 'https://raw.githubusercontent.com/HannanSeyfi/unlearning-thesis/main/Tools/github_colab_sync.py'
if not HELPER_PATH.exists():
    urllib.request.urlretrieve(HELPER_URL, HELPER_PATH)
if str(HELPER_PATH.parent) not in sys.path:
    sys.path.insert(0, str(HELPER_PATH.parent))

from github_colab_sync import commit_and_push, resolve_week35_baseline_dir, setup_colab_repo

PROJECT_ROOT = setup_colab_repo()

REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

# Non-destructive default. Set to True if you want to overwrite the canonical reports.
WRITE_CANONICAL_REPORTS = False

WEEK35_BASELINE_DIR = resolve_week35_baseline_dir(PROJECT_ROOT)
WEEK35_METRICS_PATH = (
    WEEK35_BASELINE_DIR / 'metrics.json'
    if (WEEK35_BASELINE_DIR / 'metrics.json').exists()
    else WEEK35_BASELINE_DIR / 'results' / 'metrics.json'
)
WEEK35_PERCENTAGE_PATH = (
    WEEK35_BASELINE_DIR / 'percentage_summary.csv'
    if (WEEK35_BASELINE_DIR / 'percentage_summary.csv').exists()
    else WEEK35_BASELINE_DIR / 'results' / 'percentage_summary.csv'
)

WEEK4_RESULTS_DIR = PROJECT_ROOT / 'Week 4' / 'results' / 'gradient_ascent_unlearning_v1' / 'results'
if not WEEK4_RESULTS_DIR.exists():
    WEEK4_RESULTS_DIR = PROJECT_ROOT / 'imports' / '2026-06-14-week4' / 'Week 4' / 'results' / 'gradient_ascent_unlearning_v1' / 'results'
WEEK4_METRICS_PATH = WEEK4_RESULTS_DIR / 'metrics.json'
WEEK4_PERCENTAGE_PATH = WEEK4_RESULTS_DIR / 'percentage_summary.csv'
WEEK4_HISTORY_PATH = WEEK4_RESULTS_DIR / 'unlearning_history.csv'
WEEK4_ANALYSIS_PATH = PROJECT_ROOT / 'analysis' / 'week3.5_vs_week4_results_2026-06-14.md'

paths_to_check = [
    WEEK35_METRICS_PATH,
    WEEK35_PERCENTAGE_PATH,
    WEEK4_METRICS_PATH,
    WEEK4_PERCENTAGE_PATH,
    WEEK4_HISTORY_PATH,
]

missing = [str(path) for path in paths_to_check if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required result files:\n' + '\n'.join(missing))

print(f'Project root: {PROJECT_ROOT}')
print(f'Reports directory: {REPORTS_DIR}')


In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def read_csv_dicts(path: Path):
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

def fmt_pct(value, digits=2):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "n/a"
    return f"{float(value):.{digits}f}%"

def fmt_pp(value, digits=2):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "n/a"
    sign = "+" if value > 0 else ""
    return f"{sign}{float(value):.{digits}f} pp"

def metric_pct(metrics: dict, section: str, name: str) -> float:
    """Return contains-value metric as a percentage from Week 3.5 metrics.json."""
    return 100.0 * metrics[section][name]["contains_value"]

def markdown_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(["---"] * len(headers)) + "|")
    for row in rows:
        lines.append("| " + " | ".join(str(item) for item in row) + " |")
    return "\n".join(lines)

week35_metrics = load_json(WEEK35_METRICS_PATH)
week4_metrics = load_json(WEEK4_METRICS_PATH)
week35_percentage = read_csv_dicts(WEEK35_PERCENTAGE_PATH)
week4_percentage = read_csv_dicts(WEEK4_PERCENTAGE_PATH)
week4_history = read_csv_dicts(WEEK4_HISTORY_PATH)

week35_metrics["run_name"], week4_metrics["run_name"]

## Build Summary Tables

These tables are used both for display in the notebook and for the generated Markdown reports.

In [ ]:
week35_rows = [
    ["Forget, all prompts", week35_metrics["base_before_training"]["forget_all"]["num_examples"], fmt_pct(metric_pct(week35_metrics, "base_before_training", "forget_all")), fmt_pct(metric_pct(week35_metrics, "lora_after_training", "forget_all"))],
    ["Retain, all prompts", week35_metrics["base_before_training"]["retain_all"]["num_examples"], fmt_pct(metric_pct(week35_metrics, "base_before_training", "retain_all")), fmt_pct(metric_pct(week35_metrics, "lora_after_training", "retain_all"))],
    ["Forget held-out paraphrases", week35_metrics["base_before_training"]["forget_heldout_paraphrases"]["num_examples"], fmt_pct(metric_pct(week35_metrics, "base_before_training", "forget_heldout_paraphrases")), fmt_pct(metric_pct(week35_metrics, "lora_after_training", "forget_heldout_paraphrases"))],
    ["Retain held-out paraphrases", week35_metrics["base_before_training"]["retain_heldout_paraphrases"]["num_examples"], fmt_pct(metric_pct(week35_metrics, "base_before_training", "retain_heldout_paraphrases")), fmt_pct(metric_pct(week35_metrics, "lora_after_training", "retain_heldout_paraphrases"))],
    ["Forget seen prompts", week35_metrics["base_before_training"]["forget_seen_prompts"]["num_examples"], fmt_pct(metric_pct(week35_metrics, "base_before_training", "forget_seen_prompts")), fmt_pct(metric_pct(week35_metrics, "lora_after_training", "forget_seen_prompts"))],
    ["Retain seen prompts", week35_metrics["base_before_training"]["retain_seen_prompts"]["num_examples"], fmt_pct(metric_pct(week35_metrics, "base_before_training", "retain_seen_prompts")), fmt_pct(metric_pct(week35_metrics, "lora_after_training", "retain_seen_prompts"))],
    ["General controls", week35_metrics["num_general_control_questions"], fmt_pct(week35_metrics["general_control"]["base_before_training_contains_value_percentage"]), fmt_pct(week35_metrics["general_control"]["lora_after_training_contains_value_percentage"])],
]

week35_table_md = markdown_table(["Evaluation group", "Examples", "Base before training", "LoRA after training"], week35_rows)
print(week35_table_md)

In [ ]:
before = week4_metrics["before_unlearning"]
after = week4_metrics["after_unlearning"]

week4_rows = [
    ["Forget accuracy, all", fmt_pct(before["forget_all"]), fmt_pct(after["forget_all"]), fmt_pp(after["forget_all"] - before["forget_all"])],
    ["Forget held-out paraphrases", fmt_pct(before["forget_heldout_paraphrases"]), fmt_pct(after["forget_heldout_paraphrases"]), fmt_pp(after["forget_heldout_paraphrases"] - before["forget_heldout_paraphrases"])],
    ["Retain accuracy, all", fmt_pct(before["retain_all"]), fmt_pct(after["retain_all"]), fmt_pp(after["retain_all"] - before["retain_all"])],
    ["Retain held-out paraphrases", fmt_pct(before["retain_heldout_paraphrases"]), fmt_pct(after["retain_heldout_paraphrases"]), fmt_pp(after["retain_heldout_paraphrases"] - before["retain_heldout_paraphrases"])],
    ["General controls", fmt_pct(before["general"]), fmt_pct(after["general"]), fmt_pp(after["general"] - before["general"])],
]

week4_table_md = markdown_table(["Metric", "Before unlearning", "After unlearning", "Change"], week4_rows)
print(week4_table_md)

In [ ]:
history_rows = []
for row in week4_history:
    history_rows.append([
        int(float(row["epoch"])),
        fmt_pct(float(row["forget_train_percentage"])),
        fmt_pct(float(row["retain_train_sample_percentage"])),
        "Yes" if row["retain_eligible"].strip().lower() == "true" else "No",
        f"{float(row['selection_score']):.2f}",
    ])

history_table_md = markdown_table(["Epoch", "Forget train accuracy", "Retain sample accuracy", "Eligible", "Selection score"], history_rows)
print(history_table_md)

## Generate Report Text

The next cells create Markdown strings from the loaded metrics. You can edit the text templates if your thesis wording changes.

In [ ]:
week35_report = f"""# Week 3.5 Report: High-Accuracy Learned Baseline

Generated from saved result files.

## Purpose

Week 3.5 created the learned model state used as the starting point for unlearning. The goal was to train a fresh LoRA adapter on the synthetic facts and confirm that the model had actually learned both forget and retain facts before any unlearning was attempted.

## Configuration

- Run name: `{week35_metrics['run_name']}`
- Created at UTC: `{week35_metrics['created_at_utc']}`
- Model: `{week35_metrics['model_id']}`
- Training examples: {week35_metrics['num_train_examples']}
- Forget evaluation examples: {week35_metrics['num_eval_forget_examples']}
- Retain evaluation examples: {week35_metrics['num_eval_retain_examples']}
- Held-out paraphrase prompts: {week35_metrics['num_heldout_paraphrase_prompts']}
- General-control questions: {week35_metrics['num_general_control_questions']}
- Epochs: {week35_metrics['training']['num_epochs']}
- Learning rate: {week35_metrics['training']['learning_rate']}
- 4-bit loading: {week35_metrics['training']['use_4bit']}
- Train on answer value only: {week35_metrics['train_on_fact_value_only']}
- Evaluation leakage prevented: {week35_metrics['evaluation_leakage_prevented']}

## Results

{week35_table_md}

## Interpretation

The Week 3.5 adapter learned the synthetic facts strongly. Overall forget accuracy reached {fmt_pct(metric_pct(week35_metrics, 'lora_after_training', 'forget_all'))}, and overall retain accuracy reached {fmt_pct(metric_pct(week35_metrics, 'lora_after_training', 'retain_all'))}. Performance also transferred to held-out paraphrases, which indicates the model learned more than one exact prompt wording.

The main caveat is general behavior. The saved Week 3.5 scorer reports general-control performance falling from {fmt_pct(week35_metrics['general_control']['base_before_training_contains_value_percentage'])} to {fmt_pct(week35_metrics['general_control']['lora_after_training_contains_value_percentage'])}. Later Week 4 analysis used stricter boundary-aware matching and treated the comparable pre-unlearning general score as {fmt_pct(week4_metrics['before_unlearning']['general'])}.

## Files Used

- `{WEEK35_METRICS_PATH.relative_to(PROJECT_ROOT)}`
- `{WEEK35_PERCENTAGE_PATH.relative_to(PROJECT_ROOT)}`
"""

week4_report = f"""# Week 4 Report: Gradient-Ascent Unlearning

Generated from saved result files.

## Purpose

Week 4 tested whether the Week 3.5 learned LoRA adapter could be modified to suppress the designated forget facts while preserving retain facts and general-control behavior.

## Configuration

- Run name: `{week4_metrics['run_name']}`
- Created at UTC: `{week4_metrics['created_at_utc']}`
- Base model: `{week4_metrics['base_model_id']}`
- Method: {week4_metrics['method']}
- Selected epoch: {week4_metrics['selected_epoch']}
- Maximum epochs: {week4_metrics['training']['max_epochs']}
- Learning rate: {week4_metrics['training']['learning_rate']}
- Retain weight: {week4_metrics['training']['retain_weight']}
- Batch size: {week4_metrics['training']['batch_size']}
- Gradient accumulation steps: {week4_metrics['training']['gradient_accumulation_steps']}

## Final Before/After Results

{week4_table_md}

## Checkpoint Selection

{history_table_md}

Epoch {week4_metrics['selected_epoch']} was selected because it gave substantial forgetting while keeping the retain training sample above the eligibility threshold. Later epochs reduced forget accuracy further, but retain accuracy fell too much.

## Interpretation

Gradient ascent produced substantial but incomplete forgetting. Forget accuracy fell from {fmt_pct(before['forget_all'])} to {fmt_pct(after['forget_all'])}, including a held-out paraphrase drop from {fmt_pct(before['forget_heldout_paraphrases'])} to {fmt_pct(after['forget_heldout_paraphrases'])}. However, retain accuracy also fell from {fmt_pct(before['retain_all'])} to {fmt_pct(after['retain_all'])}, so the method caused meaningful collateral damage.

The strongest supported claim is partial selective suppression with a measurable utility trade-off, not complete deletion of the targeted knowledge.

## Files Used

- `{WEEK4_METRICS_PATH.relative_to(PROJECT_ROOT)}`
- `{WEEK4_PERCENTAGE_PATH.relative_to(PROJECT_ROOT)}`
- `{WEEK4_HISTORY_PATH.relative_to(PROJECT_ROOT)}`
"""

combined_report = f"""# Week 3.5 vs Week 4 Result Summary

Generated from saved result files.

## Research Flow

Week 3.5 established that the model had learned the synthetic facts. Week 4 then applied gradient-ascent unlearning to reduce recall of the forget facts.

## Week 3.5 Learning Result

{week35_table_md}

## Week 4 Unlearning Result

{week4_table_md}

## Main Takeaway

The model learned the synthetic facts very well in Week 3.5. Week 4 then reduced forget accuracy by {fmt_pp(after['forget_all'] - before['forget_all'])}, but retain accuracy also changed by {fmt_pp(after['retain_all'] - before['retain_all'])}. This means the experiment demonstrates partial unlearning with a clear utility cost.
"""

print(week35_report[:1000])

## Write Reports

This writes the generated Markdown files into `reports`.

In [ ]:
if WRITE_CANONICAL_REPORTS:
    outputs = {
        REPORTS_DIR / "week-03-5-report.md": week35_report,
        REPORTS_DIR / "week-04-report.md": week4_report,
        REPORTS_DIR / "week-03-5-vs-week-04-summary.md": combined_report,
    }
else:
    outputs = {
        REPORTS_DIR / "week-03-5-generated-from-results.md": week35_report,
        REPORTS_DIR / "week-04-generated-from-results.md": week4_report,
        REPORTS_DIR / "week-03-5-vs-week-04-generated-summary.md": combined_report,
    }

for path, text in outputs.items():
    path.write_text(text.strip() + "\n", encoding="utf-8")
    print(f"Wrote {path}")

In [ ]:
index_lines = [
    "# Generated Week 3.5 and Week 4 Reports",
    "",
    "These files were generated by `reports/generate_week35_week4_reports.ipynb` from saved result artifacts.",
    "",
]
for path in outputs:
    index_lines.append(f"- [{path.name}]({path.name})")

index_path = REPORTS_DIR / "generated-week35-week4-index.md"
index_path.write_text("\n".join(index_lines) + "\n", encoding="utf-8")
print(f"Wrote {index_path}")

## Optional: Display Raw Tables

Use these tables if you want to inspect the underlying CSV rows directly.

In [ ]:
for name, rows in [
    ("Week 3.5 percentage summary", week35_percentage),
    ("Week 4 percentage summary", week4_percentage),
    ("Week 4 unlearning history", week4_history),
]:
    print("\n" + name)
    for row in rows:
        print(row)

In [ ]:
# GitHub output sync
commit_and_push(
    [REPORTS_DIR],
    'Colab: save generated Week 3.5 and Week 4 reports',
    repo_dir=PROJECT_ROOT,
)
